# Lie superalgebras

This notebook constructs the **basic classical Lie superalgebras**: the simple Lie superalgebras $\mathfrak g = \mathfrak g_{\bar 0} \oplus \mathfrak g_{\bar 1}$ with a reductive even part $\mathfrak g_{\bar 0}$ and an even non-degenerate invariant form [Kac1977]. They are the input of affine vertex superalgebras (`03_affine_vertex_algebras.ipynb`) and of quantum Hamiltonian reduction (`08_w_algebras.ipynb`). Sage has no Lie superalgebras, so the library also provides a category for them.

**The construction.** The contragredient Lie superalgebra $\mathfrak g(A, \tau)$ of a Cartan matrix $A$ with a set $\tau$ of odd simple roots (a super Dynkin diagram) [Kac1977] is generated by $e_i, f_i, h_i$, with $e_i, f_i$ odd for $i \in \tau$, subject to
$$[e_i, f_j] = \delta_{ij} h_i, \qquad [h_i, e_j] = a_{ij} e_j, \qquad [h_i, f_j] = -a_{ij} f_j, \qquad [h_i, h_j] = 0,$$
modulo the largest ideal that meets the Cartan subalgebra trivially. The library builds it root space by root space. For a positive root $\beta$ of height at least 2, the root space $\mathfrak g_\beta$ is spanned by the brackets $[e_i, x]$ with $x \in \mathfrak g_{\beta - \alpha_i}$, and such an element is zero exactly when $\operatorname{ad} f_j$ kills it for every $j$. So no Serre relations are needed: the extra relations at odd isotropic nodes come out by themselves. The invariant form is found by solving the invariance equations. When the Cartan matrix is singular, as for $\mathfrak{psl}(m|m)$, the algebra is also divided by its centre.

**Named types.** The named types use the distinguished Dynkin diagram, which has a single odd simple root [FSS1996]. The form is normalised as in the tables of Kac–Wakimoto [KW2004]: $(\theta|\theta) = 2$ for $\theta$ the highest root of a **chosen** simple component of the even part $\mathfrak g_{\bar 0}$. The choice is part of the name, and for $F(4)$ and $G(3)$ it is made with `theta=`. The last column is the dual Coxeter number in this normalisation, the number $h^\vee$ with $\kappa = 2h^\vee(\cdot|\cdot)$, where $\kappa$ is the super Killing form; it enters the affine vertex superalgebras through $k + h^\vee$ (`03_affine_vertex_algebras.ipynb`, section 4).

| name | parameters | even part $\mathfrak g_{\bar 0}$ | $\theta$ is the highest root of | $h^\vee$ |
|---|---|---|---|---|
| `sl(m\|n)` | $m \ge 2$, $m \ne n$ | $\mathfrak{sl}_m \oplus \mathfrak{sl}_n \oplus \mathfrak{gl}_1$ | $\mathfrak{sl}_m$ | $m - n$ |
| `psl(m\|m)` | $m \ge 2$ | $\mathfrak{sl}_m \oplus \mathfrak{sl}_m$ | $\mathfrak{sl}_m$ | $0$ |
| `osp(m\|n)` | $m \ge 4$, $n$ even | $\mathfrak{so}_m \oplus \mathfrak{sp}_n$ | $\mathfrak{so}_m$ | $m - n - 2$ |
| `spo(n\|m)` | $n \ge 2$ even | $\mathfrak{sp}_n \oplus \mathfrak{so}_m$ | $\mathfrak{sp}_n$ | $(n - m)/2 + 1$ |
| `D(2,1;a)` | $a$, given by `a=` | $\mathfrak{sl}_2 \oplus \mathfrak{sl}_2 \oplus \mathfrak{sl}_2$ | one of the $\mathfrak{sl}_2$ | $0$ |
| `F(4)`, `theta='sl2'` | — | $\mathfrak{so}_7 \oplus \mathfrak{sl}_2$ | $\mathfrak{sl}_2$ | $-2$ |
| `F(4)`, `theta='so7'` | — | $\mathfrak{so}_7 \oplus \mathfrak{sl}_2$ | $\mathfrak{so}_7$ | $3$ |
| `G(3)`, `theta='sl2'` | — | $G_2 \oplus \mathfrak{sl}_2$ | $\mathfrak{sl}_2$ | $-3/2$ |
| `G(3)`, `theta='G2'` | — | $G_2 \oplus \mathfrak{sl}_2$ | $G_2$ | $2$ |

The form depends on the choice of $\theta$ and $\kappa$ does not, so $h^\vee$ depends on it too: `osp(m|n)` and `spo(n|m)` are the same Lie superalgebra with two normalisations of the form, and so are the two rows of $F(4)$ and of $G(3)$ (section 2). Any other Dynkin diagram, or a simple Lie algebra, can be given by its Cartan matrix (section 7).

**Prerequisites:** `01_getting_started.ipynb`.

**Elsewhere:** affine vertex superalgebras are in `03_affine_vertex_algebras.ipynb`; nilpotent elements, $\mathfrak{sl}_2$-triples and the W-algebras of Lie superalgebras, including superconformal algebras, are in `08_w_algebras.ipynb`, sections 3 and 6.

**Contents**

1. A first example: $\mathfrak{sl}(2|1)$
2. The named types and Kac–Wakimoto's normalisation
3. Roots
4. The invariant form and the Killing form
5. Structure constants and the category
6. $D(2,1;a)$ with a symbolic parameter
7. Other Dynkin diagrams, Lie algebras, refusals
8. Affine vertex superalgebras
9. Unique representation and pickling
10. Limitations

In [1]:
import vertex_algebras as va
from vertex_algebras import *
from vertex_algebras.lie_superalgebras import LieSuperalgebra, LieSuperalgebras
from vertex_algebras.affine_vertex_algebra import UniversalAffineVertexAlgebra


def short(G):
    "the name of a Lie superalgebra without the base ring"
    return str(G).split(' over ')[0].replace('Lie superalgebra ', '')

## 1. A first example: $\mathfrak{sl}(2|1)$

The distinguished Dynkin diagram of $\mathfrak{sl}(2|1)$ has an even simple root $\alpha_1$ and an odd isotropic simple root $\alpha_2$; the row of the Cartan matrix of an odd isotropic simple root has a zero diagonal entry. The basis vectors are named as Sage prints Chevalley bases: `E[root]` for the root vectors, and `h1`, `h2` for the Cartan subalgebra. The cell below prints the algebra, its dimension, superdimension $\dim\mathfrak g_{\bar 0} - \dim\mathfrak g_{\bar 1}$ and rank, its Cartan matrix, the odd simple roots and the names of the basis vectors.

In [2]:
g = LieSuperalgebra(QQ, 'sl(2|1)')
print(g)
print('dimension', g.dimension(), '  superdimension', g.superdimension(), '  rank', g.rank())
print('Cartan matrix:')
print(g.cartan_matrix())
print('odd simple roots:', g.odd_simple_roots())
list(g.basis().keys())

Lie superalgebra sl(2|1) over Rational Field
dimension 8   superdimension 0   rank 2
Cartan matrix:
[ 2 -1]
[-1  0]
odd simple roots: (2,)


['E[alpha[1]]',
 'E[alpha[2]]',
 'E[alpha[1] + alpha[2]]',
 'h1',
 'h2',
 'E[-alpha[1]]',
 'E[-alpha[2]]',
 'E[-alpha[1] - alpha[2]]']

Elements are linear combinations of the basis vectors. The bracket is super-antisymmetric, $[x, y] = -(-1)^{|x||y|}[y, x]$, so $[x, y] = [y, x]$ for odd $x, y$. In the cell below, `e(i)`, `f(i)` and `h(i)` are $e_i$, $f_i$ and $h_i = [e_i, f_i]$, and `is_even_odd()` gives the parity of a homogeneous element.

In [3]:
e1, e2, f1, f2 = g.e(1), g.e(2), g.f(1), g.f(2)
print('parities:', [x.is_even_odd() for x in (e1, e2, f1, f2)])
print('[e2, f2] =', e2.bracket(f2), '   [f2, e2] =', f2.bracket(e2), '   [e2, e2] =', e2.bracket(e2))
print('[e1, e2] =', e1.bracket(e2), '   [h1, e2] =', g.h(1).bracket(e2))
x = e1 + 2*g.h(2) - 3*f2
x.bracket(e2)

parities: [0, 1, 0, 1]
[e2, f2] = h2    [f2, e2] = h2    [e2, e2] = 0
[e1, e2] = -E[alpha[1] + alpha[2]]    [h1, e2] = -E[alpha[2]]


-E[alpha[1] + alpha[2]] - 3*h2

## 2. The named types and Kac–Wakimoto's normalisation

The cell below tabulates, for a selection of named types: the dimension, the superdimension, the dual Coxeter number $h^\vee$, the normalisation root $\theta$ (in simple root coordinates) and the simple components of the even part. The values of $h^\vee$ are those of [KW2004, Tables I–III]; they are computed from the super Killing form $\kappa(x, y) = \operatorname{str}(\operatorname{ad} x \operatorname{ad} y)$, by $\kappa = 2h^\vee (\cdot|\cdot)$, and not looked up.

In [4]:
Ka.<a> = Frac(QQ['a'])
algebras = [('sl(3|2)', LieSuperalgebra(QQ, 'sl(3|2)')), ('psl(2|2)', LieSuperalgebra(QQ, 'psl(2|2)')),
            ('osp(5|2)', LieSuperalgebra(QQ, 'osp(5|2)')), ('osp(4|2)', LieSuperalgebra(QQ, 'osp(4|2)')),
            ('spo(2|3)', LieSuperalgebra(QQ, 'spo(2|3)')), ('spo(4|1)', LieSuperalgebra(QQ, 'spo(4|1)')),
            ('D(2,1;a)', LieSuperalgebra(Ka, 'D(2,1;a)', a=a)),
            ('F(4), θ in sl_2', LieSuperalgebra(QQ, 'F(4)', theta='sl2')),
            ('F(4), θ in so_7', LieSuperalgebra(QQ, 'F(4)', theta='so7')),
            ('G(3), θ in sl_2', LieSuperalgebra(QQ, 'G(3)', theta='sl2')),
            ('G(3), θ in G_2', LieSuperalgebra(QQ, 'G(3)', theta='G2'))]
print('%-17s %4s %5s %6s  %-14s %s' % ('', 'dim', 'sdim', 'h^v', 'θ', 'even part'))
for name, G in algebras:
    print('%-17s %4d %5d %6s  %-14s %s' % (name, G.dimension(), G.superdimension(), G.dual_coxeter_number(),
                                       G.theta(), ' + '.join(str(ct) for ct, _, _ in G.even_part_components())))

                   dim  sdim    h^v  θ              even part
sl(3|2)             24     0      1  (1, 1, 0, 0)   ['A', 2] + ['A', 1]
psl(2|2)            14    -2      0  (1, 0, 0)      ['A', 1] + ['A', 1]
osp(5|2)            23     3      1  (0, 1, 2)      ['B', 2] + ['A', 1]
osp(4|2)            17     1      0  (0, 0, 1)      ['A', 1] + ['A', 1] + ['A', 1]
spo(2|3)            12     0    1/2  (2, 2)         ['A', 1] + ['A', 1]
spo(4|1)            14     6    5/2  (2, 2)         ['B', 2]
D(2,1;a)            17     1      0  (2, 1, 1)      ['A', 1] + ['A', 1] + ['A', 1]
F(4), θ in sl_2     40     8     -2  (2, 3, 2, 1)   ['B', 3] + ['A', 1]
F(4), θ in so_7     40     8      3  (0, 2, 2, 1)   ['B', 3] + ['A', 1]
G(3), θ in sl_2     31     3   -3/2  (2, 4, 2)      ['G', 2] + ['A', 1]
G(3), θ in G_2      31     3      2  (0, 3, 2)      ['G', 2] + ['A', 1]


**The two normalisations of $F(4)$.** With $\theta$ in $\mathfrak{sl}_2$, the root of $\mathfrak{sl}_2$ has norm 2; with $\theta$ in $\mathfrak{so}_7$, the highest root of $\mathfrak{so}_7$ has norm 2. The cell below prints, for both choices, $h^\vee$ and the highest roots of the two components of the even part with their norms. The norms of the roots change by the factor $-\frac23$ from the second normalisation to the first. So the invariant form on $\mathfrak g$ changes by the inverse factor $-\frac32$, and since $\kappa = 2h^\vee(\cdot|\cdot)$ is fixed, $h^\vee$ changes by $-\frac23$: $-2 = -\frac23 \cdot 3$. The minimal reduction with $\theta$ in $\mathfrak{sl}_2$ gives the $N = 8$ superconformal algebra, with eight supercurrents and $\mathfrak g^\natural = \mathfrak{so}_7$ [KW2004].

In [5]:
for th in ('sl2', 'so7'):
    F = LieSuperalgebra(QQ, 'F(4)', theta=th)
    print('theta in %-4s h^v = %2s   highest roots of the components and their norms: %s'
          % (th, F.dual_coxeter_number(), [(str(ct), t, F.root_norm(t)) for ct, t, _ in F.even_part_components()]))

theta in sl2  h^v = -2   highest roots of the components and their norms: [("['B', 3]", (0, 2, 2, 1), -4/3), ("['A', 1]", (2, 3, 2, 1), 2)]
theta in so7  h^v =  3   highest roots of the components and their norms: [("['B', 3]", (0, 2, 2, 1), 2), ("['A', 1]", (2, 3, 2, 1), -3)]


## 3. Roots

Roots are given in simple root coordinates, and `root_norm` is the form on the roots induced by the normalised invariant form. Odd roots can be isotropic, that is, of norm 0. The cell below prints the Cartan matrix, the odd simple roots and $\theta$ of $\mathfrak{osp}(5|2)$, and lists its positive roots with their parities and norms; four of its five odd positive roots are isotropic.

In [6]:
G = LieSuperalgebra(QQ, 'osp(5|2)')
print('Cartan matrix:'); print(G.cartan_matrix())
print('odd simple roots:', G.odd_simple_roots(), '  theta:', G.theta())
for b in G.positive_roots():
    print('%-12s %-5s norm %s' % (b, 'odd' if b in G.odd_roots() else 'even', G.root_norm(b)))

Cartan matrix:
[ 0 -1  0]
[-1  2 -1]
[ 0 -2  2]
odd simple roots: (1,)   theta: (0, 1, 2)
(1, 0, 0)    odd   norm 0
(0, 1, 0)    even  norm 2
(0, 0, 1)    even  norm 1
(1, 1, 0)    odd   norm 0
(0, 1, 1)    even  norm 1
(1, 1, 1)    odd   norm -1
(0, 1, 2)    even  norm 2
(1, 1, 2)    odd   norm 0
(1, 2, 2)    odd   norm 0
(2, 2, 2)    even  norm -4


An odd **non-isotropic** simple root occurs in $\mathfrak{osp}(1|2)$ = `spo(2|1)`. There, $[e, e]$ is the root vector of the even root $2\alpha$, named `E[2*alpha[1]]`: the name `2*alpha[1]` makes it distinguishable from twice the vector `E[alpha[1]]`. The cell below prints the roots, their norms and $h^\vee$, the bracket $[e, e]$, and the root vectors found from roots with `root_vector`.

In [7]:
O = LieSuperalgebra(QQ, 'spo(2|1)')
e, f = O.e(1), O.f(1)
print('roots', O.roots(), '  norms', O.root_norm((1,)), O.root_norm((2,)), '  h^v', O.dual_coxeter_number())
print('[e, e] =', e.bracket(e), '   root of E[2*alpha[1]]:', O.root_on_basis('E[2*alpha[1]]'))
print('root vector of 2 alpha:', O.root_vector((2,)), '  of -alpha:', O.root_vector((1,), -1))

roots [(1,), (2,), (-1,), (-2,)]   norms 1/2 2   h^v 3/2
[e, e] = E[2*alpha[1]]    root of E[2*alpha[1]]: (2,)
root vector of 2 alpha: E[2*alpha[1]]   of -alpha: E[-alpha[1]]


## 4. The invariant form and the Killing form

`invariant_form_matrix()` is the Gram matrix of the normalised form in the order of the basis, and `invariant_form(x, y)` evaluates the form. The form is supersymmetric: $(x|y) = (-1)^{|x||y|}(y|x)$, so $(e_2|f_2) = -(f_2|e_2)$ for the odd root vectors of $\mathfrak{sl}(2|1)$. The cell below prints the Gram matrix and these values, and checks that the super Killing form is $2h^\vee$ times the normalised form on all pairs of basis vectors.

In [8]:
g = LieSuperalgebra(QQ, 'sl(2|1)')
print(g.invariant_form_matrix())
print('(e1|f1) =', g.invariant_form(g.e(1), g.f(1)), '  (e2|f2) =', g.invariant_form(g.e(2), g.f(2)),
      '  (f2|e2) =', g.invariant_form(g.f(2), g.e(2)))
B = g.basis()
all(g.killing_form(x, y) == 2 * g.dual_coxeter_number() * g.invariant_form(x, y) for x in B for y in B)

[ 0  0  0  0  0  1  0  0]
[ 0  0  0  0  0  0  1  0]
[ 0  0  0  0  0  0  0 -1]
[ 0  0  0  2 -1  0  0  0]
[ 0  0  0 -1  0  0  0  0]
[ 1  0  0  0  0  0  0  0]
[ 0 -1  0  0  0  0  0  0]
[ 0  0  1  0  0  0  0  0]
(e1|f1) = 1   (e2|f2) = 1   (f2|e2) = -1


True

For $\mathfrak{psl}(2|2)$, $h^\vee = 0$: the Killing form vanishes identically, but the invariant form is non-degenerate, as the cell below shows. So for algebras with $h^\vee = 0$ the invariant form cannot be obtained from the Killing form, and it is found by solving the invariance equations instead.

In [9]:
P = LieSuperalgebra(QQ, 'psl(2|2)')
print(P.dimension(), P.superdimension(), P.dual_coxeter_number())
print(all(P.killing_form(x, y) == 0 for x in P.basis() for y in P.basis()), P.invariant_form_matrix().det() != 0)

14 -2 0
True True


## 5. Structure constants and the category

`structure_coefficients()` lists all the nonzero brackets of basis vectors, as a dict `{(a, b): {c: coefficient}}`. This is the format in which `va.Affine` takes structure constants (`03_affine_vertex_algebras.ipynb`, section 2). The cell below prints them for $\mathfrak{osp}(1|2)$.

In [10]:
O.structure_coefficients()

{('E[alpha[1]]', 'E[alpha[1]]'): {'E[2*alpha[1]]': 1},
 ('E[alpha[1]]', 'h1'): {'E[alpha[1]]': -2},
 ('E[alpha[1]]', 'E[-alpha[1]]'): {'h1': 1},
 ('E[alpha[1]]', 'E[-2*alpha[1]]'): {'E[-alpha[1]]': -4},
 ('E[2*alpha[1]]', 'h1'): {'E[2*alpha[1]]': -4},
 ('E[2*alpha[1]]', 'E[-alpha[1]]'): {'E[alpha[1]]': -4},
 ('E[2*alpha[1]]', 'E[-2*alpha[1]]'): {'h1': -8},
 ('h1', 'E[alpha[1]]'): {'E[alpha[1]]': 2},
 ('h1', 'E[2*alpha[1]]'): {'E[2*alpha[1]]': 4},
 ('h1', 'E[-alpha[1]]'): {'E[-alpha[1]]': -2},
 ('h1', 'E[-2*alpha[1]]'): {'E[-2*alpha[1]]': -4},
 ('E[-alpha[1]]', 'E[alpha[1]]'): {'h1': 1},
 ('E[-alpha[1]]', 'E[2*alpha[1]]'): {'E[alpha[1]]': 4},
 ('E[-alpha[1]]', 'h1'): {'E[-alpha[1]]': 2},
 ('E[-alpha[1]]', 'E[-alpha[1]]'): {'E[-2*alpha[1]]': 1},
 ('E[-2*alpha[1]]', 'E[alpha[1]]'): {'E[-alpha[1]]': 4},
 ('E[-2*alpha[1]]', 'E[2*alpha[1]]'): {'h1': 8},
 ('E[-2*alpha[1]]', 'h1'): {'E[-2*alpha[1]]': 4}}

Sage has no category of Lie superalgebras: `LieAlgebras(QQ).Super()` is the category of super modules, which forgets the bracket, and the tests of Sage's Lie algebras assume antisymmetry, which fails for brackets of odd elements. The library defines the category `LieSuperalgebras(R)` over Sage's super modules. Its `TestSuite` checks super-antisymmetry, the super Jacobi identity, the parity of brackets and the invariance of the form, as the cell below shows for $\mathfrak{osp}(1|2)$.

In [11]:
print(LieAlgebras(QQ).Super())
print(LieSuperalgebras(QQ))
print(O.category())
TestSuite(O).run(verbose=True)

Category of super modules over Rational Field
Category of Lie superalgebras over Rational Field
Category of finite dimensional Lie superalgebras with basis over Rational Field
running ._test_additive_associativity() . . .

 pass


running ._test_an_element() . . .

 pass


running ._test_bracket_parity() . . .

 pass


... (45 outputs left out)

running ._test_super_jacobi_identity() . . .

 pass


running ._test_zero() . . .

 pass


## 6. $D(2,1;a)$ with a symbolic parameter

$D(2,1;a)$ is a one-parameter family of 17-dimensional Lie superalgebras whose even part is the sum of three copies of $\mathfrak{sl}_2$ [Kac1977]. The parameter $a$ appears in the Cartan matrix, and so in the brackets and the norms of the roots. The normalisation of [KW2004] puts $\theta$ in one of the three copies of $\mathfrak{sl}_2$, without saying which. The library takes the highest root $2\alpha_1 + \alpha_2 + \alpha_3$ of the distinguished positive system by default, and another root can be chosen with `theta=`. Since $h^\vee = 0$, the choice changes only the normalisation of the form, and so that of the level. The cell below works over $\mathbb Q(a)$: it prints the Cartan matrix, a bracket that involves $a$, and the highest roots of the three components with their norms, first for the default $\theta$ and then for $\theta = \alpha_2$.

In [12]:
D = LieSuperalgebra(Ka, 'D(2,1;a)', a=a)
print(D.cartan_matrix())
print('[h1, e3] =', D.h(1).bracket(D.e(3)))
print('sl_2 components (highest root, norm):', [(t, D.root_norm(t)) for _, t, _ in D.even_part_components()])
D3 = LieSuperalgebra(Ka, 'D(2,1;a)', a=a, theta=(0, 1, 0))
print('theta = alpha_2 instead:', [(t, D3.root_norm(t)) for _, t, _ in D3.even_part_components()])

[ 0  1  a]
[-1  2  0]
[-1  0  2]
[h1, e3] = a*E[alpha[3]]
sl_2 components (highest root, norm): [((0, 0, 1), -2*a/(a + 1)), ((0, 1, 0), -2/(a + 1)), ((2, 1, 1), 2)]
theta = alpha_2 instead: [((0, 0, 1), 2*a), ((0, 1, 0), 2), ((2, 1, 1), -2*a - 2)]


$D(2,1;1) \cong \mathfrak{osp}(4|2) = \mathfrak{spo}(2|4)$: the three algebras have the same dimension, superdimension, $h^\vee$ and even part, $\mathfrak{so}_4 \oplus \mathfrak{sp}_2 = \mathfrak{sl}_2^{\oplus 3}$. The norms of the roots depend on which $\mathfrak{sl}_2$ carries $\theta$:
- the default $\theta$ of $D(2,1;1)$ plays the role of the root of $\mathfrak{sp}_2$, so the norms are those of `spo(2|4)`;
- with $\theta = \alpha_2$, one of the roots of $\mathfrak{so}_4$, the norms are those of `osp(4|2)`.

The cell below compares the four algebras.

In [13]:
D1 = LieSuperalgebra(QQ, 'D(2,1;a)', a=1)
D1b = LieSuperalgebra(QQ, 'D(2,1;a)', a=1, theta=(0, 1, 0))
for name, X in [('D(2,1;1)', D1), ('spo(2|4)', LieSuperalgebra(QQ, 'spo(2|4)')),
                ('D(2,1;1), θ = α_2', D1b), ('osp(4|2)', LieSuperalgebra(QQ, 'osp(4|2)'))]:
    print('%-18s %d %d %s %s  root norms %s' % (name, X.dimension(), X.superdimension(), X.dual_coxeter_number(),
                                              [str(c) for c, _, _ in X.even_part_components()],
                                              sorted(X.root_norm(b) for b in X.positive_roots())))

D(2,1;1)           17 1 0 ["['A', 1]", "['A', 1]", "['A', 1]"]  root norms [-1, -1, 0, 0, 0, 0, 2]
spo(2|4)           17 1 0 ["['A', 1]", "['A', 1]", "['A', 1]"]  root norms [-1, -1, 0, 0, 0, 0, 2]
D(2,1;1), θ = α_2  17 1 0 ["['A', 1]", "['A', 1]", "['A', 1]"]  root norms [-4, 0, 0, 0, 0, 2, 2]
osp(4|2)           17 1 0 ["['A', 1]", "['A', 1]", "['A', 1]"]  root norms [-4, 0, 0, 0, 0, 2, 2]


## 7. Other Dynkin diagrams, Lie algebras, refusals

A Lie superalgebra can have several inequivalent Dynkin diagrams, related by odd reflections [FSS1996], and any of them can be given by its Cartan matrix together with the odd simple roots (`odd=`). For a Cartan matrix, `theta=` gives the normalisation root; it is required when the highest root is odd. The cell below gives two examples, and compares each with the corresponding named type:
- $\mathfrak{sl}(2|1)$ with both simple roots odd;
- $\mathfrak{osp}(3|2)$ with the simple roots $\varepsilon - \delta$ (odd isotropic) and $\delta$ (odd non-isotropic), against the named `spo(2|3)`.

In [14]:
H = LieSuperalgebra(QQ, matrix([[0, 1], [1, 0]]), odd=[1, 2], theta=(1, 1))
print(H)
print(H.dimension(), H.superdimension(), H.dual_coxeter_number(), '  as sl(2|1):', g.dimension(), g.superdimension(), g.dual_coxeter_number())
O32 = LieSuperalgebra(QQ, matrix([[0, 1], [-2, 2]]), odd=[1, 2], theta=(0, 2))
S23 = LieSuperalgebra(QQ, 'spo(2|3)')
[(X.dimension(), X.superdimension(), X.dual_coxeter_number(), [str(c) for c, _, _ in X.even_part_components()])
 for X in (O32, S23)]

Lie superalgebra with Cartan matrix [[0, 1], [1, 0]] and odd simple roots [1, 2] over Rational Field
8 0 1   as sl(2|1): 8 0 1


[(12, 0, 1/2, ["['A', 1]", "['A', 1]"]),
 (12, 0, 1/2, ["['A', 1]", "['A', 1]"])]

Without odd simple roots, the construction gives the simple Lie algebras, normalised as in [KW2004, Table I], with the long roots of norm 2. The cell below compares $h^\vee$ with Sage's values for the types $B_3$, $G_2$, $F_4$ and $E_6$.

In [15]:
for t, r in [('B', 3), ('G', 2), ('F', 4), ('E', 6)]:
    L = LieSuperalgebra(QQ, CartanMatrix([t, r]))
    print('%s%d: dimension %3d, h^v %2d (Sage: %2d)' % (t, r, L.dimension(), L.dual_coxeter_number(),
                                                     CartanType([t, r]).dual_coxeter_number()))

B3: dimension  21, h^v  5 (Sage:  5)
G2: dimension  14, h^v  4 (Sage:  4)


F4: dimension  52, h^v  9 (Sage:  9)


E6: dimension  78, h^v 12 (Sage: 12)


**Refusals.** The cell below shows five inputs that the constructor refuses with a `ValueError`: $F(4)$ without a choice of normalisation; `osp(3|2)`, since the name `osp(m|n)` needs $m \ge 4$ (use `spo(2|3)`); `sl(2|2)`, which is not simple (use `psl(2|2)`); a Cartan matrix of affine type, whose Lie algebra is infinite-dimensional; and a Cartan matrix whose highest root is odd, given without `theta=`.

In [16]:
for args, kw in [(('F(4)',), {}), (('osp(3|2)',), {}), (('sl(2|2)',), {}),
                 ((matrix([[2, -2], [-2, 2]]),), {}), ((matrix([[2, -1], [-1, 0]]),), {'odd': [2]})]:
    try:
        LieSuperalgebra(QQ, *args, **kw)
    except ValueError as err:
        print('ValueError:', err)

ValueError: F(4) has two normalisations of the form: give theta='sl2' or theta='so7'
ValueError: osp(m|n) has θ in so_m: m >= 4, n even (and so_4 is not simple); use spo(n|m) for θ in sp_n
ValueError: sl(m|m) is not simple: use psl(m|m)
ValueError: the Lie superalgebra of this Cartan matrix is not finite dimensional (roots of height above 100)
ValueError: the highest root is odd: give theta (a highest root of a simple component of the even part)


## 8. Affine vertex superalgebras

`UniversalAffineVertexAlgebra(R, g, level=k)` takes a Lie superalgebra (see `03_affine_vertex_algebras.ipynb`). It uses the names of the basis vectors as generator names, the level times the normalised form as the second-order poles, and the grading by the root lattice. The constructor checks the super Jacobi identity and the invariance of the form, and builds the Sugawara vector, whose central charge is $c = k \operatorname{sdim}\mathfrak g/(k + h^\vee)$ [KRW2003]. The cell below checks this central charge for seven Lie superalgebras over $\mathbb Q(k)$.

In [17]:
K.<k> = Frac(QQ['k'])
for G in [LieSuperalgebra(QQ, 'spo(2|1)'), LieSuperalgebra(QQ, 'sl(2|1)'), LieSuperalgebra(QQ, 'psl(2|2)'),
          LieSuperalgebra(QQ, 'osp(4|2)'), LieSuperalgebra(QQ, 'spo(4|1)'),
          LieSuperalgebra(QQ, 'F(4)', theta='sl2'), LieSuperalgebra(QQ, 'G(3)', theta='sl2')]:
    V = UniversalAffineVertexAlgebra(K, G, level=k, conformal_vector='sugawara')
    expected = k * G.superdimension() / (k + G.dual_coxeter_number())
    print('%-10s c = %-14s %s' % (short(G), V.central_charge(), V.central_charge() == expected))

spo(2|1)   c = k/(k + 3/2)    True
sl(2|1)    c = 0              True
psl(2|2)   c = -2             True
osp(4|2)   c = 1              True


spo(4|1)   c = 6*k/(k + 5/2)  True


F(4)       c = 8*k/(k - 2)    True


G(3)       c = 3*k/(k - 3/2)  True


The cell below shows the currents of $V^k(\mathfrak{osp}(1|2))$, which satisfy $x(z) y(w) \sim k(x|y)/(z-w)^2 + [x, y](w)/(z-w)$ and are graded by the root lattice. For the odd root vector $x = e$, $(x|x) = 0$, so $x(z) x(w) \sim [x, x](w)/(z-w)$.

In [18]:
V = UniversalAffineVertexAlgebra(K, O, level=k, conformal_vector='sugawara')
print(V)
print(V.variable_names())
x, y = V.gen('E[alpha[1]]'), V.gen('E[-alpha[1]]')
print('x x:', x.ope(x), '   x y:', x.ope(y), '   grade of x:', x.horizontal_grade())
print(V.lie_algebra() is O)

Universal affine vertex superalgebra of Lie superalgebra spo(2|1) over Rational Field at level k over Fraction Field of Univariate Polynomial Ring in k over Rational Field
('E[alpha[1]]', 'E[2*alpha[1]]', 'h1', 'E[-alpha[1]]', 'E[-2*alpha[1]]')
x x: [E[2*alpha[1]]]    x y: [h1, 4*k]    grade of x: (1)
True


As a vector space, the universal affine vertex superalgebra is the supersymmetric algebra on the negative modes of the currents, so its character is $\prod_{n \ge 1} (1+q^n)^{\dim \mathfrak g_{\bar 1}} / (1-q^n)^{\dim \mathfrak g_{\bar 0}}$. For $\mathfrak{osp}(1|2)$ this is $\prod_{n \ge 1}(1+q^n)^2/(1-q^n)^3$, which the cell below checks up to $q^7$.

In [19]:
ch = V.character(8)
R.<q> = PowerSeriesRing(QQ, default_prec=8)
expected = prod((1 + q^n)^2 / (1 - q^n)^3 for n in range(1, 8))
print(ch)
L = ch.laurent_part()
print(all(L[n] == expected[n] for n in range(8)))

1 + 5*q + 18*q^2 + 55*q^3 + 149*q^4 + 371*q^5 + 867*q^6 + 1923*q^7 + O(q^8)
True


For $D(2,1;a)$ over $\mathbb Q(a, k)$, $\operatorname{sdim} = 9 - 8 = 1$ and $h^\vee = 0$, so $c = k/k = 1$ for every $a$ and $k$:

In [20]:
Kak.<aa, kk> = Frac(QQ['aa', 'kk'])
Da = LieSuperalgebra(Frac(QQ['aa']), 'D(2,1;a)', a=Frac(QQ['aa']).gen())
UniversalAffineVertexAlgebra(Kak, Da, level=kk, conformal_vector='sugawara').central_charge()

1

## 9. Unique representation and pickling

Equal data give the same object, and pickles come back as the same object, as the cell below checks for Lie superalgebras and for an affine vertex superalgebra.

In [21]:
print(g is LieSuperalgebra(QQ, 'sl(2|1)'), loads(dumps(g)) is g, loads(dumps(D)) is D, loads(dumps(H)) is H)
print(loads(dumps(V)) is V)

True True True True
True


## 10. Limitations

* **Basic classical Lie superalgebras only.** The construction needs a Cartan matrix of finite type, so the strange Lie superalgebras $\mathfrak p(n)$ and $\mathfrak q(n)$ and the Lie superalgebras of Cartan type are not available. Lie superalgebras that are not simple, such as $\mathfrak{gl}(m|n)$ or $\mathfrak{sl}(m|m)$, are refused by name; they can still be given to the affine constructor by structure constants and an invariant form (`03_affine_vertex_algebras.ipynb`, section 2).
* **Finiteness test.** A Cartan matrix is declared not of finite type when the construction finds roots of height above 100.
* **Dynkin diagrams by hand.** The named types use the distinguished diagram. Other diagrams must be entered as Cartan matrices; odd reflections are not computed, and isomorphisms between the Lie superalgebras of different diagrams are not constructed (sections 6 and 7 compare invariants only).
* **Normalisations.** For $D(2,1;a)$, $F(4)$ and $G(3)$ the normalisation of the form depends on the choice of $\theta$, and with it $h^\vee$ (for $F(4)$ and $G(3)$) and the meaning of the level.
* **No representation theory.** Modules over Lie superalgebras, their characters and Weyl groups (or Weyl groupoids) are not implemented; the library uses Lie superalgebras as input for vertex algebras.

## References

- [Kac1977] V. G. Kac, *Lie superalgebras*, Adv. Math. 26 (1977), 8–96, [doi:10.1016/0001-8708(77)90017-2](https://doi.org/10.1016/0001-8708(77)90017-2). — basic classical Lie superalgebras, the contragredient construction, $D(2,1;a)$ (introduction, section 6).
- [FSS1996] L. Frappat, A. Sciarrino, P. Sorba, *Dictionary on Lie superalgebras*, preprint (1996), [arXiv:hep-th/9607161](https://arxiv.org/abs/hep-th/9607161). — the distinguished Dynkin diagrams of the named types, other Dynkin diagrams and odd reflections (introduction, section 7).
- [KW2004] V. G. Kac, M. Wakimoto, *Quantum reduction and representation theory of superconformal algebras*, Adv. Math. 185 (2004), 400–458; corrigendum, Adv. Math. 193 (2005), 453–455, [arXiv:math-ph/0304011](https://arxiv.org/abs/math-ph/0304011). — the normalisation of the invariant forms and the values of $h^\vee$ (introduction, sections 2, 6, 7).
- [KRW2003] V. G. Kac, S.-S. Roan, M. Wakimoto, *Quantum reduction for affine superalgebras*, Comm. Math. Phys. 241 (2003), 307–342, [arXiv:math-ph/0302015](https://arxiv.org/abs/math-ph/0302015). — the Sugawara vector of affine vertex superalgebras and its central charge (section 8).